# 第 16 章互动实验：对齐、亚像素与周期图案

所有估计都调用你在 `16-1_相位相关与亚像素.py` 里的实现。先预测结果，再运行。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "16_图像对齐与亚像素"
ASSIGNMENT = "16-1_相位相关与亚像素.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def texture(shape=(96, 96), seed=0, width=0.08):
    rng = np.random.default_rng(seed)
    f = np.fft.fft2(rng.normal(size=shape))
    fy = np.fft.fftfreq(shape[0])[:, None]; fx = np.fft.fftfreq(shape[1])[None, :]
    return np.real(np.fft.ifft2(f * np.exp(-(fy**2 + fx**2) / (2 * width**2))))

def fourier_shift(img, dy, dx):
    fy = np.fft.fftfreq(img.shape[0])[:, None]; fx = np.fft.fftfreq(img.shape[1])[None, :]
    return np.real(np.fft.ifft2(np.fft.fft2(img) * np.exp(-2j * np.pi * (fy * dy + fx * dx))))

def run_or_hint(experiment):
    """运行一个实验函数；它用到的作业函数还没完成时，打印提示并返回 None。"""
    try:
        return experiment()
    except NotImplementedError as exc:
        print(f"先完成作业：{exc}")
        return None

ref = texture()
y, x = np.mgrid[0:96, 0:96]
periodic = ((x % 8) < 4).astype(float) + 0.5 * (((y % 8) < 4))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(ref, cmap="gray"); ax[0].set_title("随机纹理")
ax[1].imshow(periodic, cmap="gray"); ax[1].set_title("周期 8 px 的图案")
plt.show()

## 实验 1：亚像素误差随小数位移的变化

预测：误差会不会随小数部分呈周期性变化？最大误差在 0.25 还是 0.5 附近？

In [ ]:
fracs = np.linspace(0, 1, 21)
errs = []
for f in fracs:
    est = attempt(mod.phase_correlation, ref, fourier_shift(ref, 3 + f, -2 - f), use_window=False)
    if est is None:
        break
    errs.append((est[0] - (3 + f), est[1] - (-2 - f)))
if errs:
    errs = np.array(errs)
    plt.plot(fracs[:len(errs)], errs[:, 0], "o-", label="dy 误差")
    plt.plot(fracs[:len(errs)], errs[:, 1], "s-", label="dx 误差")
    plt.axhline(0, color="k", lw=0.5); plt.xlabel("真值的小数部分"); plt.ylabel("误差 (px)"); plt.legend(); plt.show()
    print("最大绝对误差", np.abs(errs).max().round(3), "px")

## 实验 2：周期图案的假峰

周期 8 px。平移 3 px 和平移 5 px 时，估计结果分别是多少？为什么 5 px 会出问题？

In [ ]:
for true in [(0, 3), (0, 5), (0, 9)]:
    est = attempt(mod.phase_correlation, periodic, np.roll(periodic, true, axis=(0, 1)), use_window=False)
    if est is not None:
        print(f"真值 {true} → 估计 ({est[0]:.2f}, {est[1]:.2f})")
print("对照第 16 章教程第 8 节：怎样用对位标记或限制搜索范围解决？实验 7 会试其中一种。")

## 实验 3：ZNCC 对亮度变化的不变性

In [ ]:
img = (ref - ref.min()) / np.ptp(ref) * 200
tpl = img[30:50, 40:64]
dim = img * 0.6 + 30
for name, target in [("原图", img), ("变暗加偏置", dim)]:
    m = attempt(mod.ncc_map, target, tpl)
    if m is None:
        break
    py, px = np.unravel_index(np.argmax(m), m.shape)
    print(f"{name}：峰值位置 ({py}, {px})，峰值 {m.max():.4f}")
if m is not None:
    plt.imshow(m, cmap="magma"); plt.colorbar(); plt.title("ZNCC 响应图"); plt.show()
if cv2 is not None:
    r = cv2.matchTemplate(img.astype(np.float32), tpl.astype(np.float32), cv2.TM_CCOEFF_NORMED)
    print("OpenCV TM_CCOEFF_NORMED 峰值位置", np.unravel_index(np.argmax(r), r.shape))

## 实验 4：噪声与加窗

真实图像边界不连续。下面用“裁剪”而不是循环平移制造位移，比较加窗和不加窗。

In [ ]:
big = texture((160, 160), seed=3)
rng = np.random.default_rng(1)
rows = []
for use_window in (False, True):
    e = []
    for _ in range(20):
        dy, dx = rng.integers(-6, 7, size=2)
        a = big[30:126, 30:126]
        b = big[30 - dy:126 - dy, 30 - dx:126 - dx] + rng.normal(scale=0.05 * big.std(), size=(96, 96))
        est = attempt(mod.phase_correlation, a, b, use_window=use_window)
        if est is None:
            break
        e.append(max(abs(est[0] - dy), abs(est[1] - dx)))
    if e:
        rows.append((use_window, np.mean(e), np.max(e)))
for w, mean_e, max_e in rows:
    print(f"加窗={w}：平均误差 {mean_e:.3f} px，最大 {max_e:.3f} px")

## 实验 5：同一个开关，换一种数据

实验 4 的两幅图是从大图里裁出来的，边界不连续。这次只改一处：第二幅图用 `np.roll` 循环平移得到，其余（纹理、噪声、平移范围）都不变。验收里的大部分用例、第 24 章的合成数据都属于循环平移这一类。

预测：在循环平移的数据上，加窗让误差变小还是变大？和实验 4 的结论一样吗？写下理由再运行。

In [ ]:
def window_experiment(trials=40):
    big = texture((160, 160), seed=3)
    a = big[30:126, 30:126]
    rows = []
    for cyclic in (False, True):
        for use_window in (False, True):
            rng = np.random.default_rng(1)
            e = []
            for _ in range(trials):
                dy, dx = rng.integers(-6, 7, size=2)
                b = np.roll(a, (dy, dx), axis=(0, 1)) if cyclic else big[30 - dy:126 - dy, 30 - dx:126 - dx]
                b = b + rng.normal(scale=0.05 * big.std(), size=(96, 96))
                est = mod.phase_correlation(a, b, use_window=use_window)
                e.append(max(abs(est[0] - dy), abs(est[1] - dx)))
            e = np.array(e)
            rows.append(("循环平移" if cyclic else "裁剪视场", use_window, np.median(e), e.max(), int((e > 1).sum())))
    return rows

rows = run_or_hint(window_experiment)
if rows:
    for kind, w, med_e, max_e, n_bad in rows:
        print(f"{kind}，加窗={w}：误差中位数 {med_e:.3f} px，最大 {max_e:.2f} px，超过 1 px 的有 {n_bad}/40 次")
    print("想一想：真实相机拍到的图更像哪一种？只用循环平移的数据验证算法，会漏掉什么问题？（教程第 7 节）")

## 实验 6：把估计出的平移移回去，再估一次

实验 1 的曲线说明三点抛物线有系统偏差，偏差的大小随真值的小数部分变化。教程第 6.3 节的第一种改进是迭代：按当前估计把第二幅图移回去，对剩下的残差再做一次相位相关，把两次结果相加。下面的 `refine` 只调用你的 `phase_correlation`，没有新的算法。

预测：移回去之后，残差的小数部分更接近 0 还是更接近 0.5？对照实验 1 的曲线，这时抛物线的偏差是大还是小？迭代两轮后最大误差大约降到原来的几分之一？

In [ ]:
def refine(ref_img, mov_img, rounds):
    """rounds=0 是只做一次相位相关；每多一轮，就把 mov_img 按当前估计移回去，再估一次残差。"""
    total = np.array(mod.phase_correlation(ref_img, mov_img, use_window=False))
    for _ in range(rounds):
        back = fourier_shift(mov_img, -total[0], -total[1])
        total = total + np.array(mod.phase_correlation(ref_img, back, use_window=False))
    return total

def refine_experiment():
    fracs = np.linspace(0, 1, 21)
    curves = {}
    for rounds in (0, 1, 2):
        errs = []
        for f in fracs:
            est = refine(ref, fourier_shift(ref, 3 + f, -2 - f), rounds)
            errs.append(max(abs(est[0] - (3 + f)), abs(est[1] - (-2 - f))))
        curves[rounds] = np.array(errs)
    return fracs, curves

result = run_or_hint(refine_experiment)
if result:
    fracs, curves = result
    for rounds, errs in curves.items():
        plt.plot(fracs, errs, "o-", ms=3, label=f"迭代 {rounds} 轮：最大 {errs.max():.3f} px")
    plt.xlabel("真值的小数部分"); plt.ylabel("误差（dy、dx 中较大者，px）"); plt.legend(); plt.show()
    print("最大误差：", "，".join(f"迭代 {rounds} 轮 {errs.max():.3f} px" for rounds, errs in curves.items()))
    print("想一想：这里的第二幅图是无噪声的循环平移。换成实验 4 的裁剪视场，“移回去”这一步还能这样做吗？")

## 实验 7：给周期图案加一个对位标记

实验 2 里，周期图案上的平移一旦超过半个周期就估错。在图案左上角叠一个不重复的十字，其余不变，两幅图各加一点噪声，随机平移 40 次。

预测：十字占的面积不到图像的百分之一，它能把结果救回来吗？没有标记时，40 次里大约错多少次，错的时候是不是正好错了整数个周期？

In [ ]:
marked = periodic.copy()
marked[11:13, 5:20] += 1.5
marked[5:20, 11:13] += 1.5

def mark_experiment(trials=40):
    out = {}
    for name, img in (("没有标记", periodic), ("有十字标记", marked)):
        rng = np.random.default_rng(4)
        errs = []
        for _ in range(trials):
            true = rng.integers(-7, 8, size=2)
            a = img + rng.normal(scale=0.02, size=img.shape)
            b = np.roll(img, true, axis=(0, 1)) + rng.normal(scale=0.02, size=img.shape)
            est = mod.phase_correlation(a, b, use_window=False)
            errs.append((est[0] - true[0], est[1] - true[1]))
        out[name] = np.abs(np.array(errs))
    return out

result = run_or_hint(mark_experiment)
if result:
    for name, errs in result.items():
        wrong = errs[errs.max(axis=1) > 1]
        whole = int((np.abs(wrong - 8 * np.round(wrong / 8)) < 0.5).all(axis=1).sum())
        print(f"{name}：40 次里误差超过 1 px 的有 {len(wrong)} 次，其中两个方向都正好错了整数个周期（8 px 的整数倍）的有 {whole} 次")
    plt.imshow(marked, cmap="gray"); plt.title("带十字对位标记的周期图案"); plt.show()
    print("想一想：教程第 8 节讲了周期图案上的两种失败。没有标记时这里是哪一种，还是两种都有？十字的频谱和周期图案的频谱有什么不同？")

## 破坏实验与记录

- 把噪声加到纹理标准差的 50%，误差怎么变？
- 把纹理换成更平滑的（width=0.02），相位相关还可靠吗？
- 把实验 7 的十字改小（例如只留 3×3 的一个点），或者把噪声加大 10 倍，标记还管用吗？
- 把实验 1 的曲线、实验 2 的失败案例、实验 5 的四行结果写进 `实验记录.md`。